# Chapter 6 — Credit Risk: discrimination metrics (companion demo)

Deterministic, no network. Reproduces the credit-scoring discrimination metrics
(KS statistic, AUROC, Gini) and the disparate-impact ratio defined in the chapter, on a
small seeded synthetic score set. The full UCI German Credit lab is in `exercises.ipynb`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)  # seeded -> deterministic

# Synthetic scores: defaulters (y=1) score higher on average than non-defaulters.
n_pos, n_neg = 300, 700
s_pos = np.clip(rng.normal(0.62, 0.18, n_pos), 0, 1)
s_neg = np.clip(rng.normal(0.40, 0.18, n_neg), 0, 1)
scores = np.concatenate([s_pos, s_neg])
y = np.concatenate([np.ones(n_pos), np.zeros(n_neg)])

In [ ]:
# AUROC = P(S+ > S-) via the Mann-Whitney U statistic.
order = np.argsort(scores)
ranks = np.empty_like(order, dtype=float); ranks[order] = np.arange(1, len(scores) + 1)
auroc = (ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

# KS = max vertical gap between the two empirical CDFs.
grid = np.linspace(0, 1, 501)
F_pos = np.searchsorted(np.sort(s_pos), grid, side='right') / n_pos
F_neg = np.searchsorted(np.sort(s_neg), grid, side='right') / n_neg
ks = np.max(np.abs(F_pos - F_neg))
gini = 2 * auroc - 1
print(f'AUROC = {auroc:.4f}   KS = {ks:.4f}   Gini = {gini:.4f}')

In [ ]:
# Disparate-impact ratio (four-fifths rule): approval rates by protected attribute.
approve_protected, approve_reference = 0.42, 0.58
dir_ratio = approve_protected / approve_reference
print(f'DIR = {dir_ratio:.3f}  ->', 'FLAG (<0.8)' if dir_ratio < 0.8 else 'OK')